# Method: Multilevel Perceptron

### I. Imports and Reads

#### Imports

In [34]:
import numpy as np
import csv 

#### Reads

In [35]:
with open('../dataset/processed_test.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    test_in = np.array(data, dtype=np.float64)
    test_out = np.array(output, dtype=np.float64)

with open('../dataset/processed_tune.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    tune_in = np.array(data, dtype=np.float64)
    tune_out = np.array(output, dtype=np.float64)

with open('../dataset/processed_train.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    train_in = np.array(data, dtype=np.float64)
    train_out = np.array(output, dtype=np.float64)

### II. Setup

#### Initial Parameters


In [36]:
learning_rate = 0.0001
input_size = train_in.shape[1]
N = train_in.shape[0]
batch_size = N
layer_sizes = [16,16,16] + [1]

total_iters = 0

np.random.seed(1)

#### Activation Functions

In [37]:
def ReLU(layers, leaky = 0):
    if type(layers) == list:
        layers = np.array(layers, dtype=np.float64)
    return np.where(layers < 0, layers * leaky, layers)

def ReLUDerivative(layers, leaky = 0):
    return np.where(layers < 0, leaky, 1)

#### Layer Setup

In [38]:
weights = [np.random.randn(input_size, layer_sizes[0])]
biases = []
layers = [np.zeros((batch_size, input_size), dtype=np.float64)]
for x in layer_sizes:
    biases.append(np.ones((x), dtype=np.float64))
    layers.append(np.zeros((batch_size, x), dtype=np.float64))

for i in range(0, len(layer_sizes)-1):
    weights.append(np.random.randn(layer_sizes[i], layer_sizes[i+1]) * ((2/layer_sizes[i]) ** 0.5))

#https://machinelearningmastery.com/weight-initialization-for-deep-learning-neural-networks/

print([x.shape for x in layers])
print([x.shape for x in weights])
print([x.shape for x in biases])

[(3174, 6), (3174, 16), (3174, 16), (3174, 16), (3174, 1)]
[(6, 16), (16, 16), (16, 16), (16, 1)]
[(16,), (16,), (16,), (1,)]


### III. Training

In [39]:
iters = 600000
learning_rate = 0.000001
leaky = 0
slow_rate = True
for iter in range(iters):
    # get batch_size datapoints
    cost = np.zeros((batch_size))
    indexes = np.random.choice(range(N), size = batch_size, replace= False)
    layers[0] = train_in[indexes]
    expected = train_out[indexes].reshape(-1, 1)

    #forward prop
    for i in range(len(layers)-1):
        if i < len(layers)-1: 
            layers[i+1] = ReLU(layers[i].dot(weights[i])+biases[i], leaky = leaky)
        else:
            layers[i+1] = layers[i].dot(weights[i])+biases[i]

    cost = (layers[-1] - expected)**2

    # print(cost)
    if iter % 1000 == 0:
        print(f"Mean Squared Error [{total_iters}]: {cost.mean()}")
    # backward prop

    bias_diff = [np.zeros_like(x) for x in biases]
    weights_diff = [np.zeros_like(x) for x in weights]

    delta = 2 * (layers[-1] - expected)
    for i in range(len(layer_sizes) - 1, -1, -1):
        weights_diff[i] = layers[i].T @ delta / batch_size
        bias_diff[i] = delta.mean(axis = 0)

        if i > 0:
            delta = delta @ weights[i].T * ReLUDerivative(layers[i], leaky = leaky)

    for x in range(len(layer_sizes)-1, -1, -1):
            weights[x] = weights[x] - learning_rate * weights_diff[x]
            biases[x] = biases[x] - learning_rate * bias_diff[x]

    total_iters += 1


Mean Squared Error [0]: 1744.3206718966603
Mean Squared Error [1000]: 7.690864871528579
Mean Squared Error [2000]: 6.31957212372045
Mean Squared Error [3000]: 5.681956589078377
Mean Squared Error [4000]: 5.305900994083463
Mean Squared Error [5000]: 5.046077807941615
Mean Squared Error [6000]: 4.849420957206922
Mean Squared Error [7000]: 4.692140914643148
Mean Squared Error [8000]: 4.562877642176573
Mean Squared Error [9000]: 4.453746561214391
Mean Squared Error [10000]: 4.360176652071713
Mean Squared Error [11000]: 4.279048818257429
Mean Squared Error [12000]: 4.208025873114624
Mean Squared Error [13000]: 4.145245446381334
Mean Squared Error [14000]: 4.0894474705705335
Mean Squared Error [15000]: 4.039679052531425
Mean Squared Error [16000]: 3.9949775118441657
Mean Squared Error [17000]: 3.9545829468922657
Mean Squared Error [18000]: 3.9179560240688645
Mean Squared Error [19000]: 3.8846201373753932
Mean Squared Error [20000]: 3.854226179135702
Mean Squared Error [21000]: 3.826444796203

### IV. Test Weights on a Dataset

In [40]:
layers[0] = test_in
expected = test_out.reshape(-1, 1)

#forward prop
for i in range(len(layers)-1):
    if i < len(layers)-1: 
        layers[i+1] = ReLU(layers[i].dot(weights[i])+biases[i], leaky = leaky)
    else:
        layers[i+1] = layers[i].dot(weights[i])+biases[i]

cost = (layers[-1] - expected)**2


SStotal = (expected - expected.mean())**2
SStotal = SStotal.sum()

SSres = (layers[-1] - expected)**2
SSres = SSres.sum()

print(f"RMSE [test]: {((layers[-1] - expected)**2).mean() ** 0.5}")
print(f"MAE [test]: {(layers[-1] - expected).mean()}")
print(f"MAPE [test]: {np.abs((expected - layers[-1])/expected).mean()*100}%")
print(f"R2 [test]: {1 - (SSres/SStotal)}")


# get batch_size datapoints
layers[0] = tune_in
expected = tune_out.reshape(-1, 1)

#forward prop
for i in range(len(layers)-1):
    if i < len(layers)-1: 
        layers[i+1] = ReLU(layers[i].dot(weights[i])+biases[i], leaky = leaky)
    else:
        layers[i+1] = layers[i].dot(weights[i])+biases[i]

cost = (layers[-1] - expected)**2

# print(cost)
SStotal = (expected - expected.mean())**2
SStotal = SStotal.sum()

SSres = (layers[-1] - expected)**2
SSres = SSres.sum()


print(f"\nRMSE [tune]: {((layers[-1] - expected)**2).mean() ** 0.5}")
print(f"MAE [tune]: {(layers[-1] - expected).mean()}")
print(f"MAPE [tune]: {np.abs((expected - layers[-1])/expected).mean()*100}%")
print(f"R2 [tune]: {1 - (SSres/SStotal)}")

RMSE [test]: 2.4479424120825173
MAE [test]: 0.754502743713605
MAPE [test]: 4.03254313935583%
R2 [test]: 0.7482446046149128

RMSE [tune]: 2.2185174770324405
MAE [tune]: 0.9991730817726441
MAPE [tune]: 4.026904208372546%
R2 [tune]: 0.7587912398465142


### V. Save current weights

## DONT FORGET TO RE-COMMENT AFTER SAVING TO PREVENT LOSS OF DATA

In [41]:
np.savez("MLP_weights4", *weights)
np.savez("MLP_biases4", *biases)